In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = '1'
import numpy as np
import pandas as pd
from typing import Dict, List
# from dotenv import load_dotenv  # pip install python-dotenv
from dynclim.features.safety import SegmentationConfig, ThresholdConfig, CompositeConfig 
from dynclim.experiments.runner import ExperimentRunner
from dynclim.data import RailDataPipeline
from dynclim.utils import loadDomainsFromYaml
import arrow
# from dynclim.data.database import get_reports_from_db, create_database_engine
# load_dotenv(dotenv_path='/dynClimToolkit/.env')
# load_dotenv()
from pathlib import Path
# If your notebook is in the project root
project_root = Path("/Users/mrosen44/Documents/dev/DynamicNarrativeClimateToolkit")



2026-02-04 14:50:26.365 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-02-04 14:50:26.367 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-02-04 14:50:26.368 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


In [2]:
# prepare domains and data for feature extraction
domain_definitions = loadDomainsFromYaml(project_root / 'config/domains/safety_domains.yaml')

# Get data for feature extraction
pipeline = RailDataPipeline(
    events_path=project_root / "dynclim/data/raw/rail/Rail_Equipment_Accident_Incident_Data_(Form_54)_20260204.csv",
    output_dir=project_root / "dynclim/data/processed/rail"
)
results = pipeline.run()
report_data = pipeline.get_report_data()  # [(eid, narrative), ...]

2026-02-04 14:50:30,266 - INFO - ============================================================
2026-02-04 14:50:30,267 - INFO - Running Rail Data Pipeline
2026-02-04 14:50:30,267 - INFO - ============================================================
2026-02-04 14:50:30,267 - INFO - Loading events from /Users/mrosen44/Documents/dev/DynamicNarrativeClimateToolkit/dynclim/data/raw/rail/Rail_Equipment_Accident_Incident_Data_(Form_54)_20260204.csv
2026-02-04 14:50:34,215 - INFO - Loaded 223,691 raw event records
2026-02-04 14:50:34,325 - WARNING - No raw event ID column found, using index
2026-02-04 14:50:34,670 - INFO - Generated UUIDs for 223,691 events
2026-02-04 14:50:35,010 - INFO - Using existing narrative column: 'Narrative'
2026-02-04 14:50:35,184 - INFO - Using existing date column: 'Date'
2026-02-04 14:50:36,926 - INFO - Cleaning: 223,691 -> 143,535 events (80,156 removed)
2026-02-04 14:50:37,725 - INFO - Saved 143,535 records to /Users/mrosen44/Documents/dev/DynamicNarrativeClimate

In [ ]:
# Run experiment with checkpointing

runner = ExperimentRunner(
    domain_definitions = domain_definitions,
    use_acceleration=True,
    device_pref='auto',
    use_parallel=False,
    embedding_model_names=[      # first three are from same model family but differ in size
                                 # last option is more performant in general, but a different model family
        'all-MiniLM-L6-v2',      # small, from sentence transformer team
        # 'all-mpnet-base-v2',     # medium, from sentence transformer team
        'all-roberta-large-v1',  # large, from sentence transformer team
        'BAAI/bge-large-en-v1.5' # very large, and generally better performance, but has different training method
                                 # when compared to above models which vary just on size
        ]
)

# Test with subset of configurations
config_filter = {
    #'segmentation_methods': ['sentence'],
    #'threshold_methods': ['top_k'],
    #'composite_methods': ['attention_weight', 'weighted_average'],
    'sentiment_methods': ['roberta', 'distilbert'] #, 'finbert','vader']
}

results_df = runner.run_experiment(
    report_data[:100], 
    experiment_name = "rail_test_02-04-2026",
    config_filter=config_filter,
    resume_from_checkpoint=True,
    # save_interval=2,
    chunk_size=None
    # max_workers=10
)